lo que pudimos hacer con esta prueba de examen fue construir un agente que sea capaz de jugar el juego de cruzar la calle o autopista sin ser arrollado o chocado, en si chocado porque seria como un humano o asi. Para esto lo que hice fue construir un entorno, en si una matriz de 5x5, en la que se encuentren los autos y el agente. los estados en si que el agentr va a tomar es de espera, derecha, izq. la poltica es que no choque, el fin es de que no choque con los atos e intete cruzar lo antes posible, y se tiene un catigos de -100 puntos y recompensa de 100, cada vez que se sube el numero de priebas el agente toma una mejor estrategia para jugar, ademas de que para eso se uso qlearning

In [14]:
import numpy as np
import math
import random
import time
from IPython.display import clear_output 

class EntornoSemaforo:
    def __init__(self, ancho=5, alto=5):
        self.ancho = ancho
        self.alto = alto
        # aca decimos en donde es que los coches van a estar, o sea en que carril en si va aser su posicion inicial
        self.coches = [
            {"y": 1, "x_inicial": 0, "dir": 1},   
            {"y": 2, "x_inicial": 4, "dir": -1},  
            {"y": 3, "x_inicial": 2, "dir": 1}    
        ]
        self.reset()

    def reset(self):
        # reseteamos la posicion del agente si es que el agente ya llego a jugar una aprtida anterior
        self.agente_pos = [self.ancho // 2, 0] 
        self.tiempo = 0
        return self._get_estado()

    def _get_estado(self):
        fase_coches = self.tiempo % self.ancho # El ciclo se repite cada 'ancho' pasos
        return (self.agente_pos[0], self.agente_pos[1], fase_coches)

    def acciones_validas(self):
        return ['ARRIBA', 'ABAJO', 'IZQUIERDA', 'DERECHA', 'ESPERAR']

    def step(self, accion):
        x, y = self.agente_pos
        
        # movemos el agente, o bueno se mueve en esas posiciones
        if accion == 'ARRIBA' and y < self.alto - 1: y += 1
        elif accion == 'ABAJO' and y > 0: y -= 1
        elif accion == 'IZQUIERDA' and x > 0: x -= 1
        elif accion == 'DERECHA' and x < self.ancho - 1: x += 1
        
        self.agente_pos = [x, y]
        self.tiempo += 1
        
        # comprobamos choques del agente
        chocado = False #iniclizanos en falso ya que obviamente al principio el agente no ha chocado
        for c in self.coches:
            pos_x_coche = (c["x_inicial"] + c["dir"] * self.tiempo) % self.ancho
            if self.agente_pos[1] == c["y"] and self.agente_pos[0] == pos_x_coche:
                chocado = True
                break

        # aca decimos cuales y cuantos sersn su castigos o recompesas  por cada accion que tome
        estado_siguiente = self._get_estado()
        if chocado:
            return estado_siguiente, -100, True # le restamos -100 por haber chocado tal cual
        elif self.agente_pos[1] == self.alto - 1:
            return estado_siguiente, 100, True  # si el agente llgego a la meta que era estar en la posicion mas alta le
            #damos una recompensa de 100 puntos
        else:
            return estado_siguiente, -1, False  # si el agente no llegara a realiar algun movimiento tambien lo castigamos 
            #pero no con tanta cantidad

    def render(self):
        print(f"--- Tiempo: {self.tiempo} ---")
        for y in range(self.alto - 1, -1, -1):
            fila = ""
            for x in range(self.ancho):
                coche_aqui = False
                for c in self.coches:
                    pos_x_coche = (c["x_inicial"] + c["dir"] * self.tiempo) % self.ancho
                    if c["y"] == y and pos_x_coche == x:
                        coche_aqui = True
                        break
                
                if self.agente_pos == [x, y]:
                    fila += "[A]" # agente
                elif coche_aqui:
                    fila += "[C]" # coche
                elif y == self.alto - 1:
                    fila += " M " # meta
                else:
                    fila += " . " # calle
            print(fila)
        print("-------------------\n")

In [15]:
class Agente:
    def __init__(self, alpha=0.1, gamma=0.9, c=2.0):
        self.Q = {}  
        self.N = {}  
        self.alpha = alpha
        self.gamma = gamma
        self.c = c

    def get_q(self, estado, accion):
        return self.Q.get((estado, accion), 0.0)

    def elegir_accion(self, estado, acciones, entrenando=True):
        if not entrenando:
            # i ya no estamos entrenando sera explotacion
            mejores = [a for a in acciones if self.get_q(estado, a) == max([self.get_q(estado, act) for act in acciones])]
            return random.choice(mejores)

        n_estado = sum(self.N.get((estado, a), 0) for a in acciones)
        if n_estado == 0:
            return random.choice(acciones)

        mejor_accion = None
        max_ucb = -float('inf')

        for a in acciones:
            q_val = self.get_q(estado, a)
            n_sa = self.N.get((estado, a), 0)

            if n_sa == 0: return a
            
            bonus = self.c * math.sqrt(math.log(n_estado) / n_sa)
            ucb_val = q_val + bonus

            if ucb_val > max_ucb:
                max_ucb = ucb_val
                mejor_accion = a

        return mejor_accion

    def actualizar(self, estado, accion, recompensa, estado_siguiente, acciones_sig, fin):
        self.N[(estado, accion)] = self.N.get((estado, accion), 0) + 1
        q_actual = self.get_q(estado, accion)
        
        if fin:
            q_objetivo = recompensa
        else:
            q_max_siguiente = max([self.get_q(estado_siguiente, a) for a in acciones_sig])
            q_objetivo = recompensa + self.gamma * q_max_siguiente
            
        self.Q[(estado, accion)] = q_actual + self.alpha * (q_objetivo - q_actual)


In [19]:
entorno = EntornoSemaforo()
agente = AgenteUCB(alpha=0.2, gamma=0.9, c=2.5)
acciones = entorno.acciones_validas()

episodios_entrenamiento = 10000
print("Entrenando al agente... por favor espera.")

for i in range(episodios_entrenamiento):
    estado = entorno.reset()
    fin = False
    
    pasos = 0 
    while not fin and pasos < 50:
        accion = agente.elegir_accion(estado, acciones, entrenando=True)
        estado_siguiente, recompensa, fin = entorno.step(accion)
        
        agente.actualizar(estado, accion, recompensa, estado_siguiente, acciones, fin)
        estado = estado_siguiente
        pasos += 1

print("¡Entrenamiento finalizado!\n")

Entrenando al agente... por favor espera.
¡Entrenamiento finalizado!



In [22]:
print("Iniciando simulación visual del Agente entrenado...")
time.sleep(2)

estado = entorno.reset()
fin = False

while not fin:
    clear_output(wait=True)
    entorno.render()
    
    accion = agente.elegir_accion(estado, acciones, entrenando=False)
    print(f"Acción elegida por el agente: {accion}")
    time.sleep(0.8) 
    
    estado, recompensa, fin = entorno.step(accion)

clear_output(wait=True)
entorno.render()
if recompensa == 100:
    print("¡El agente ha cruzado la calle de forma segura!")
else:
    print("El agente chocó :(")

--- Tiempo: 6 ---
 M  M  M  M [A]
 .  .  . [C] . 
 .  .  . [C] . 
 . [C] .  .  . 
 .  .  .  .  . 
-------------------

¡El agente ha cruzado la calle de forma segura!
